## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [43]:
load_dotenv(override=True)

True

In [44]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

#openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
ollama_api_key = "OLLAMA"

In [189]:

from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel

ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

deepseek_model = OpenAIChatCompletionsModel(model="deepseek-r1",
 openai_client=ollama_client)
mistral_model = OpenAIChatCompletionsModel( 
    model="mistral-small:latest",
    openai_client=ollama_client
)
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.5-flash", openai_client=gemini_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)

gpt_oss_model = OpenAIChatCompletionsModel(model="gpt-oss:20b", openai_client=ollama_client)
gpt_oss_120b_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)
qwen3_30b_model = OpenAIChatCompletionsModel(model="gpt-oss:20b", openai_client=ollama_client)

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 2.5 cents per call.

Costs are here: https://platform.openai.com/docs/pricing#web-search

In [215]:
from langchain_community.tools import DuckDuckGoSearchResults, DuckDuckGoSearchRun

@function_tool
def search(search_string: str) -> str:
    search = DuckDuckGoSearchRun()
    search_result = search.invoke(search_string)
    print(search_result)
    return search_result


In [216]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[search],
    model=mistral_model,
    model_settings=ModelSettings(tool_choice="required"),
)

In [217]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

/Users/dennistreder-tschechlov/Documents/Workspace/udemy/agents/.venv/lib/python3.12/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Breaking News, Latest News and Current News from FOXNews.com. Breaking news and video. Latest Current News: U.S., World, Entertainment, Health, Business, Technology, Politics, Sports. Viele übersetzte Beispielsätze mit "at the latest" – Deutsch-Englisch Wörterbuch und Suchmaschine für Millionen von Deutsch-Übersetzungen. View the latest news and breaking news today for U.S., world, weather, entertainment, politics and health at CNN.com. Top Designer Fashion bei Hechler&Nickel online bestellen oder direkt im Store vorbeischauen - ENTDECKE die LATEST TRENDS und TOP BRANDS. Get all of the latest breaking local and international news stories as they happen, with up to the minute updates and analysis, from Ireland's National Broadcaster


AI Agent frameworks in 2025.

1-LangChain & AutoGPT.
- LangChain: Modular framework for AI applications, integrating LLMs with databases, APIs, file systems
 - AutoGPT: Enhances LangChain using advanced prompting techniques

2- Agentic:
  - Combines language and action to enable real-world interaction by breaking down tasks into smaller steps

3- AutoGen.
  - Multi-agent framework for enabling coordination between language models to solve complex problems that individual LLMs cannot on their own.
   - Each agent can assume specialized roles, such as a planner or an executor
  4- Toolformer: Allows language models with access to tools and APIs for enhanced functionality

5-CAMEL:  Conveys agents multiple experts with a single assistant as a facilitator. It also enables collaboration using natural language through task allocation.

6. Reflexion: Enables training-less generalization by enabling language models to refine responses in iterative manner utilizing feedback from the environment.
 - Utilizes natural language prompts

7-GOD:  Combines retrieval methods and generative AI.
 - Enhances problem-solving capabilities when dealing with complex and dynamic environments
 - Facilitates seamless integration between AI agents and real-world tasks

### As always, take a look at the trace

https://platform.openai.com/traces

### We will now use Structured Outputs, and include a description of the fields

In [218]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model=mistral_model,
    output_type=WebSearchPlan,
)

In [219]:

message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='To find out any AI agent framework created around 2024-2025', query='New AI Agent Frameworks 2025\n'), WebSearchItem(reason='To get the latest updates on the current popular AI frameworks.', query='Most Latest AI Agent models\n'), WebSearchItem(reason='To find out if there are AI frameworks that are developed by a major company such as Google, Microsoft or OpenAI', query='AI Agent Frameworks 2025 developed by Major Companies')]


In [157]:
[search.query for search in result.final_output.searches]

['AI Agent development frameworks and vendors future trends',
 'AI Agent technology roadmap 2025']

In [220]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_sender = os.getenv('FROM_EMAIL')
    to_recipient = os.getenv('TO_EMAIL') 
    print(f"Sending from email: {from_sender[0:10]}")
    from_email = Email(from_sender) # Change this to your verified email
    to_email = To(to_recipient) # Change this to your email
    print(f"to sender {to_recipient[0:10]}: ")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [221]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model=qwen3_30b_model,
)



In [222]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first think about an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report - so fill the outline with content - and return that as your final output.\n"
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model=gemini_model,
    output_type=ReportData,
)

message = "Latest Topics and Advancement in Agentic AI in 2025"
with trace("Writer"):
    result = await Runner.run(writer_agent, message)
    print(result.final_output)


short_summary='In 2025, Agentic AI will see significant advancements driven by more capable LLMs, sophisticated multi-agent systems, and integrated embodied intelligence. Key developments include self-reflective architectures, continuous learning paradigms, and transformative applications in autonomous software development, scientific discovery, and personalized services. While offering immense potential, critical challenges around safety, interpretability, and ethical governance remain paramount for responsible deployment.' markdown_report='# The Horizon of Autonomy: Latest Topics and Advancements in Agentic AI in 2025\n\n## 1. Introduction\n\nArtificial Intelligence has rapidly evolved beyond mere pattern recognition and prediction, ushering in an era where AI systems are not just tools but increasingly autonomous entities capable of goal-oriented behavior, planning, and self-correction. This paradigm shift defines Agentic AI – systems designed to perceive their environment, make dec

In [206]:
display(Markdown(result.final_output.markdown_report))

# The Dawn of Autonomous Intelligence: Latest Topics and Advancements in Agentic AI by 2025

## I. Executive Summary

By 2025, Agentic AI is poised to transcend its foundational large language model (LLM) roots, evolving into sophisticated, goal-driven systems capable of increasingly autonomous action, complex problem-solving, and seamless human collaboration. Key advancements will focus on enhancing agent planning, multi-agent coordination, robust safety mechanisms, and real-world embodiment, transforming various industries by automating intricate tasks and augmenting human capabilities. This report details the projected landscape of Agentic AI, highlighting the technological breakthroughs and critical challenges anticipated by mid-decade.

## II. Introduction: The Rise of Agentic AI

The landscape of Artificial Intelligence is undergoing a profound transformation, moving beyond passive predictive models and conversational interfaces towards systems capable of active, goal-oriented behavior. This paradigm shift defines the emergence of **Agentic AI**—intelligent entities designed not merely to respond to prompts but to autonomously identify problems, formulate plans, execute actions, and continuously adapt within dynamic environments. Unlike traditional AI applications focused on narrow tasks, agentic systems aim for a broader, more integrated form of intelligence, mirroring aspects of human cognitive processes such as perception, reasoning, memory, planning, and action.

Historically, AI agents existed in rudimentary forms, from rule-based expert systems to early planning algorithms. However, the advent of powerful Large Language Models (LLMs) in the early 2020s provided the foundational cognitive engine necessary to unlock true agentic capabilities. LLMs bestow agents with unprecedented natural language understanding, reasoning, and generation abilities, enabling them to comprehend complex instructions, generate intricate plans, interact with humans, and communicate with other digital systems through natural language interfaces. This report delves into the projected advancements and pivotal topics shaping Agentic AI by 2025, anticipating a period of rapid evolution and significant real-world deployment. The focus is on forward-looking developments, inferring trends from current research trajectories and industry investments to paint a picture of what these intelligent agents will be capable of in the near future.

## III. Foundational Pillars Driving Agentic AI in 2025

The projected advancements in Agentic AI by 2025 are firmly built upon the continued maturation of several core technological pillars. These foundational elements are not static but are themselves undergoing rapid evolution, collectively empowering agents with greater cognitive prowess and operational autonomy.

### A. Large Language Models (LLMs) as the Cognitive Core:
By 2025, LLMs will solidify their role as the primary cognitive engine for agentic systems. Their capabilities will extend far beyond basic text generation:

1.  **Enhanced Reasoning and Planning Capabilities:** Future LLMs will exhibit significantly improved logical reasoning, mathematical capabilities, and hierarchical planning abilities. This means agents will be able to break down complex, multi-step goals into smaller, manageable sub-tasks, anticipate consequences of actions, and generate more robust and efficient plans. Techniques like Chain-of-Thought (CoT) and Tree-of-Thought (ToT) prompting, combined with self-correction mechanisms, will be more deeply integrated into the LLM architecture itself, leading to more reliable reasoning.
2.  **Multimodality and Embodied Understanding:** LLMs in 2025 will be inherently multimodal, seamlessly processing and generating information across text, image, audio, and potentially even tactile data. This multimodal understanding is critical for agents to perceive and interact with the physical world, understanding visual cues, spatial relationships, and environmental sounds. This will enable agents to interpret richer contextual information, leading to more nuanced decision-making in diverse environments.
3.  **Context Window Expansion and Long-term Memory Integration:** While current LLMs have limited context windows, 2025 will see significant advancements in managing and recalling information over much longer durations. This will involve hybrid approaches combining massive context windows with sophisticated retrieval augmentation generation (RAG) techniques and external memory systems. Agents will be able to remember past interactions, learned facts, and personal preferences over extended periods, making their behavior more consistent, personalized, and informed.

### B. Advanced Tool Use and API Orchestration:
One of the defining characteristics of agentic AI is its ability to use external tools. By 2025, this capability will become far more sophisticated:

1.  **Dynamic Tool Selection and Composition:** Agents will move beyond predefined tool lists, intelligently selecting and dynamically composing tools (e.g., APIs, internal functions, web search engines, code interpreters) based on the task at hand and the current environment. This involves sophisticated tool-calling mechanisms that can understand tool specifications, handle complex inputs/outputs, and manage tool dependencies.
2.  **Integration with External Knowledge Bases and Real-time Data:** Agents will seamlessly integrate with vast external knowledge bases (e.g., specialized databases, scientific literature, proprietary company data) and real-time data streams (e.g., financial markets, sensor data). This allows agents to access up-to-the-minute information, enhancing their decision-making and problem-solving capabilities in dynamic scenarios.

### C. Memory Systems: Beyond Short-Term Context:
Memory is crucial for continuous learning and intelligent behavior. By 2025, agents will incorporate advanced memory architectures:

1.  **Hierarchical Memory Architectures:** Agents will employ sophisticated memory systems that mimic aspects of human memory, including: 
    *   **Episodic Memory:** Storing specific past experiences, events, and interactions, allowing agents to recall 

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [223]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    print(f"Searching for {input}")
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and email it

In [224]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.short_summary)
    print("Email sent")
    return report

### Showtime!

In [225]:
import warnings
warnings.filterwarnings(action="ignore")

query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    print(search_plan)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")



Starting research...
Planning searches...
Will perform 3 searches
searches=[WebSearchItem(reason='To keep the list specific and relevant.', query='Latest AI agent frameworks 2024 or after.'), WebSearchItem(reason='For potential upcoming innovations.', query='Upcoming AI Agent frameworks 2023 or after.'), WebSearchItem(reason='AI Agents for 2025 can be compared effectively with 2024 trends. So, knowing what was popular in the past should guide us to what we are looking for now', query='Popular AI agent frameworks 2024')]
Searching...
Searching for Search term: Latest AI agent frameworks 2024 or after.
Reason for searching: To keep the list specific and relevant.
Searching for Search term: Upcoming AI Agent frameworks 2023 or after.
Reason for searching: For potential upcoming innovations.
Searching for Search term: Popular AI agent frameworks 2024
Reason for searching: AI Agents for 2025 can be compared effectively with 2024 trends. So, knowing what was popular in the past should guide 

/Users/dennistreder-tschechlov/Documents/Workspace/udemy/agents/.venv/lib/python3.12/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Jun 27, 2025 · As we look towards 2026, the frameworks underpinning this transformation are poised for significant evolution. Jul 10, 2025 · Frameworks like LangChain, AutoGen, CrewAI, and LlamaIndex remain prominent tools for orchestrating agents, while workflow platforms like Zapier and n8n are making agent … 2 days ago · Latest AI News: August 14, 2025 Stay informed with our daily curated artificial intelligence news and insights. We bring you the most important updates in AI, machine … Feb 19, 2025 · 🌟 The Multi-Agent Framework: First AI Software Company, Towards Natural Language Programming - FoundationAgents/MetaGPT Jul 30, 2025 · I reviewed several popular open-source AI agent frameworks. In this article, I break down each framework’s multi-agent orchestration capabilities, agent and function definitions, …


/Users/dennistreder-tschechlov/Documents/Workspace/udemy/agents/.venv/lib/python3.12/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


No good DuckDuckGo Search Result was found
No good DuckDuckGo Search Result was found


...

Mar 6, 2025 · The collective, called AGNTCY, aims to create a standard way for AI agents from different platforms and frameworks to talk to each other. May 1, 2025 · Explore top AI agent frameworks and how to choose the right platform to build intelligent, scalable systems that deploy smarter autonomous agents. Apr 8, 2025 · Restack AI SDK The framework for AI agents Build reliable and accurate AI agents in code, capable of running and persisting month-lasting processes in the background. Jul 30, 2025 · I reviewed several popular open-source AI agent frameworks. In this article, I break down each framework’s multi-agent orchestration capabilities, agent and function definitions, … Apr 24, 2025 · Agentic AI frameworks let machines think, act, and improve on their own. This overview compares LangChain, Auto-GPT, Semantic Kernel, and others. It covers key features, …


/Users/dennistreder-tschechlov/Documents/Workspace/udemy/agents/.venv/lib/python3.12/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Jun 9, 2024 · popular的用法和搭配1. popular的定义与用法：popularity作为名词，表示“流行”，“人气”，“受欢迎”， … Aug 16, 2019 · popular的用法2：popular引申可作“通俗的”“大众 (化)的”解,指适合一般人的爱好,需要或在一般人能理解的范围内,多含 … be popular to意思和be popular with差不多,但是用法上有点区别 例如：If you want to impress your peers (friends) it is … Popular的名词是Popularity。 重点词汇： 1，Popular adj. 流行的，通俗的；受欢迎的；大众的；普及的 双语例句： This is one … 1. be popular in：基本意思是“流行的”“大众喜爱的”，指受到大部分人所欢迎和喜爱的，作此解时，可用作定语，也可用作表语，有 …


/Users/dennistreder-tschechlov/Documents/Workspace/udemy/agents/.venv/lib/python3.12/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/Users/dennistreder-tschechlov/Documents/Workspace/udemy/agents/.venv/lib/python3.12/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


199元AI模型永久会员，该团队已经跑路了，模型已于2023年就已经停止更新，现在是被欧艺接手，但人家只放出个 欧艺7.0 供已够买永久会员来使用，最新的模型是 欧艺8.0，两者并不相 … 目前的 AI大模型本质是，以统计规律代替逻辑规律，以相关性代替因果性，以海量参数的函数拟合出输入输出算法。 具体的技术原理：1 是通过数据集获得统计规律，2是通过统计规律进行插 … 知乎，中文互联网高质量的问答社区和创作者聚集的原创内容平台，于 2011 年 1 月正式上线，以「让人们更好的分享知识、经验和见解，找到自己的解答」为品牌使命。知乎凭借认真、专业 … 1）Cursor Cursor是国外一家公司在2023年发布的一款AI原生的IDE，主要功能就是通过AI能力自动检索理解代码的上下文、可以自动编写并运行终端命令、可以自动检测并修正代码、具有强 … 接着是32B的模型，这个模型对显存的要求至少要20GB，这时候 AI甜品卡RTX 4060 TI 16GB也爆显存了，RTX 4060更是雪上加霜，一秒吐不出一个字。 这时候核显的优势就显现出来了，在 …
Python的错误提示“most recent call last”中，last是什么意思？ 什么用法？ 这句话中的每个单词都认识，但是连在一起就搞不懂了，能理解大概意思是“最近的一次调用”，可是为什么在最后还 … Gabung dan ikuti diskusi perkembangan pasar modal bersama komunitas investor dan trader Mandiri Sekuritas 介绍华硕B760主板型号的详细信息，帮助用户了解其特点和适用性。 most与most of的区别 一、most后可直接跟名词 (可数或不可数)，同时，也可接有形容词修饰的名词，跟可数名词时， 谓语动词 要用复数形式。 二、most后不能直接跟有定冠词、 指示代词  … most of the后面可以接名词单数，也可以接可数名词复数。 1、most of the +可数名词单数， 谓语动词 要用单数形式。例如： Most of the apple is on the table. 那只苹果的大部分在桌子上。 2 …
Finished searching
Thinking about report...
Fi

In [105]:
report

ReportData(short_summary='This query is about researching latest AI frameworks as an expert. It involves outlining a report based on search results covering emerging technologies in 2025.', markdown_report='# Comprehensive Report: Latest AI Agent Frameworks in 2025\\n\\n## Outline\\n1. **Introduction**\\n   - Define what AI agents are and their importance\\n   - Explain the purpose of using frameworks for development\\n2. **Current Standalone Framework Focus (April–October 2024)**\\n   - Present individual framework features with examples\\n3. **Cross‑Modal Infrastructure Breakthroughs (Mid-Year Updates)**\\n   - Discuss interoperability initiatives like AGNTCY standard\\n4. **Industry Trends in AI Development**\\n   - Analyze scalability, hardware integration, and modularity trends\\n5. **Future Outlook: Interoperability & Multi-Agent Systems**\\n   - Explore potential advancements and impacts\\n6. **Case Studies/Examples Across Industries (2024–mid 2025)**\\n7. **Conclusion\\n## Fina

In [ ]:
from IPython.display import display, Markdown
display(Markdown(report.short_summary))

# Outline\n**### Report Title:** Latest Developments and Trends in AI Agent Frameworks \n## Structure and Flow: \n* Introduction to the topic \n  * Definition of AI agents \n  * Context - recent advancements (pre-2025) and current trends. \n* Section on GPT-based agents \n  * Describe their architecture, functionality, strengths, limitations.\n* Section on Autogen framework as an example of advanced agent systems \n  * Overview of the framework and its evolution/updates?\n*', 

### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>